# transforms

In [1]:
from torchvision import transforms
from PIL import Image
from torch.utils.tensorboard import SummaryWriter
import cv2

# 1. 常见 transforms 工具的使用

In [17]:
image_path = r"hymenoptera_data_2\train\ants_image\0013035.jpg"
image = Image.open(image_path) # 读取图片
type(image)

PIL.JpegImagePlugin.JpegImageFile

In [16]:
image_cv = cv2.imread(image_path)
type(image_cv)

numpy.ndarray

## （1）张量化

In [3]:
tensor_trans = transforms.ToTensor() # 实例化
image_tensor = tensor_trans(image) # 将图片转换成tensor类型
print(image_tensor)

tensor([[[0.3137, 0.3137, 0.3137,  ..., 0.3176, 0.3098, 0.2980],
         [0.3176, 0.3176, 0.3176,  ..., 0.3176, 0.3098, 0.2980],
         [0.3216, 0.3216, 0.3216,  ..., 0.3137, 0.3098, 0.3020],
         ...,
         [0.3412, 0.3412, 0.3373,  ..., 0.1725, 0.3725, 0.3529],
         [0.3412, 0.3412, 0.3373,  ..., 0.3294, 0.3529, 0.3294],
         [0.3412, 0.3412, 0.3373,  ..., 0.3098, 0.3059, 0.3294]],

        [[0.5922, 0.5922, 0.5922,  ..., 0.5961, 0.5882, 0.5765],
         [0.5961, 0.5961, 0.5961,  ..., 0.5961, 0.5882, 0.5765],
         [0.6000, 0.6000, 0.6000,  ..., 0.5922, 0.5882, 0.5804],
         ...,
         [0.6275, 0.6275, 0.6235,  ..., 0.3608, 0.6196, 0.6157],
         [0.6275, 0.6275, 0.6235,  ..., 0.5765, 0.6275, 0.5961],
         [0.6275, 0.6275, 0.6235,  ..., 0.6275, 0.6235, 0.6314]],

        [[0.9137, 0.9137, 0.9137,  ..., 0.9176, 0.9098, 0.8980],
         [0.9176, 0.9176, 0.9176,  ..., 0.9176, 0.9098, 0.8980],
         [0.9216, 0.9216, 0.9216,  ..., 0.9137, 0.9098, 0.

In [5]:
writer = SummaryWriter("logs")
writer.add_image("image_tensor", image_tensor) # 往logs中添加图片

## （2）归一化

In [ ]:
print("before:", image_tensor[0][0][0])
trans_norm = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # 传入均值和标准差的列表，注意每一个channel都要有
image_norm = trans_norm(image_tensor) # 注意输入要为tensor类型
print("after:", image_norm[0][0][0])

before: tensor(0.3137)
after: tensor(-0.3725)


In [7]:
writer.add_image("Norm", image_norm) # 往logs中添加图片

## （3）改变形状

In [8]:
print("before:", image.size)
trans_resize = transforms.Resize((512, 512)) # 输入一个元组就是指定形状
image_resize = trans_resize(image)
print("after:", image_resize.size)

before: (768, 512)
after: (512, 512)


In [9]:
image_resize_tensor = tensor_trans(image_resize) # 先转换成tensor类型，再保存进日志
writer.add_image("Resize", image_resize_tensor, 0)

## （4）工具组合

In [10]:
trans_resize_2 = transforms.Resize(468) # 输入一个值就是等比缩放
trans_compose = transforms.Compose([trans_resize_2, tensor_trans]) # 将这两个工具组合成工作流
image_compose = trans_compose(image) # 输入为第一个工具的输入

In [11]:
writer.add_image("Compose", image_compose, 0)

## （5）随机裁剪

In [12]:
trans_randcrop = transforms.RandomCrop(256) # 输入一个数x就按照(H, W)=(x, x)来剪切
trans_compose_2 = transforms.Compose([trans_randcrop, tensor_trans])

In [13]:
for i in range(10):
    image_crop = trans_compose_2(image)
    writer.add_image("RandCrop_2", image_crop, i)

In [14]:
print("Finish！")
writer.close() # 写完之后，记得close()

Finish！


# 2. 与数据集结合

In [2]:
import torchvision

*以torchvision自带的CIFAR10数据集为例*

In [3]:
train_set = torchvision.datasets.CIFAR10(root="./my_cifar10", train=True, download=True) # download=True下载数据集，train=True训练集
test_set = torchvision.datasets.CIFAR10(root="./my_cifar10", train=False, download=True) # train=False测试集

100.0%


*下载慢的话也可以通过迅雷下载，再将压缩包(.gz)复制到./my_cifar10文件夹中，再运行上面代码*

In [5]:
test_set.classes

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [4]:
test_set[0]

(<PIL.Image.Image image mode=RGB size=32x32>, 3)

*PIL是指数据（图片），数字3是指第4个标签（也就是cat）*

In [6]:
image, target = test_set[0]
image.show()

In [8]:
image.size

(32, 32)

In [7]:
test_set.classes[target]

'cat'

- 与transforms结合

In [ ]:
trans_dataset = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

In [20]:
# transform=trans_dataset读取时对每一个图片进行自己设置的transforms变换
train_set_trans = torchvision.datasets.CIFAR10(root="./my_cifar10", train=True, transform=trans_dataset, download=True)
test_set_trans = torchvision.datasets.CIFAR10(root="./my_cifar10", train=False, transform=trans_dataset, download=True)

In [23]:
test_set_trans[0]

(tensor([[[0.6196, 0.6235, 0.6471,  ..., 0.5373, 0.4941, 0.4549],
          [0.5961, 0.5922, 0.6235,  ..., 0.5333, 0.4902, 0.4667],
          [0.5922, 0.5922, 0.6196,  ..., 0.5451, 0.5098, 0.4706],
          ...,
          [0.2667, 0.1647, 0.1216,  ..., 0.1490, 0.0510, 0.1569],
          [0.2392, 0.1922, 0.1373,  ..., 0.1020, 0.1137, 0.0784],
          [0.2118, 0.2196, 0.1765,  ..., 0.0941, 0.1333, 0.0824]],
 
         [[0.4392, 0.4353, 0.4549,  ..., 0.3725, 0.3569, 0.3333],
          [0.4392, 0.4314, 0.4471,  ..., 0.3725, 0.3569, 0.3451],
          [0.4314, 0.4275, 0.4353,  ..., 0.3843, 0.3725, 0.3490],
          ...,
          [0.4863, 0.3922, 0.3451,  ..., 0.3804, 0.2510, 0.3333],
          [0.4549, 0.4000, 0.3333,  ..., 0.3216, 0.3216, 0.2510],
          [0.4196, 0.4118, 0.3490,  ..., 0.3020, 0.3294, 0.2627]],
 
         [[0.1922, 0.1843, 0.2000,  ..., 0.1412, 0.1412, 0.1294],
          [0.2000, 0.1569, 0.1765,  ..., 0.1216, 0.1255, 0.1333],
          [0.1843, 0.1294, 0.1412,  ...,

In [ ]:
# 写入日志
writer_set = SummaryWriter("logs_dataset")

for i in range(10):
    image, target = test_set_trans[i]
    writer_set.add_image("test_set", image, i)

writer_set.close()